In [ ]:
import os
import mne
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from scipy import signal
from glob import glob
from mne_bids import BIDSPath, read_raw_bids
from scipy.stats import ttest_rel
import math
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
%matplotlib qt

### define inputs

In [ ]:
# file paths
bids_root = '/Users/xjf5193/Library/CloudStorage/OneDrive-NorthwesternUniversity/SoundBrain Lab - EAM1/data-bids'
events_dir = '/Users/xjf5193/Library/CloudStorage/OneDrive-NorthwesternUniversity/PhD/Soundbrain lab/EAM/stimtrack-compare/python_v2'

# epochs can be generated based on either the status events or the stim-track-generated events
# choose trigger codes for epoching: 'ergo1' for stim-track-generated events, 'Status' for trigger codes from the status channel, or 'both' for both sets of events (will generate two sets of epochs)
EPOCHING_TRIGGER_SOURCE = 'both'  # options: 'ergo1', 'status', 'both'

# derivatives from status
deriv_dir_motor = os.path.join(bids_root, 'derivatives', 'cortical_filtered_motor')
if not os.path.exists(deriv_dir_motor):
    os.makedirs(deriv_dir_motor)

deriv_dir_status = os.path.join(bids_root, 'derivatives', 'cortical_filtered_status')
if not os.path.exists(deriv_dir_status):
    os.makedirs(deriv_dir_status) 

# derivatives from stimtrack
deriv_dir = os.path.join(bids_root, 'derivatives', 'cortical_filtered')
if not os.path.exists(deriv_dir):
    os.makedirs(deriv_dir)

# Preprocess data (BIDS)

In [ ]:
task_list = ['active', 'passive', 'motor']
bl_start = 0.100
l_freq = 1
h_freq = 30
passive_hex = '#4477AA'
active_hex = '#CC6677'

### Cortically filtered (1–30 Hz) epochs

In [ ]:

# generate epochs for all participants and tasks, and save them in the derivatives folder
for sub_num in range(2, 35):
    sub_label = f'{sub_num:02d}'

    for task_label in task_list:
        print(f'Loading {task_label} data')

        epoch_list_ergo1 = []
        epoch_list_status = []

        for run_label in range(1,6):
            # load in EEG data
            bids_path = BIDSPath(root=bids_root,
                                 datatype='eeg',
                                 subject=str(sub_label), 
                                 task=task_label, 
                                 run=run_label)
            
            try:
                data = read_raw_bids(bids_path, verbose=False)
                data.load_data(verbose=False)

                # re-reference data to linked mastoid reference
                data_ref = data.set_eeg_reference(ref_channels=['M1', 'M2'], verbose=False)
                data_filtered = data_ref.copy().filter(l_freq=l_freq, h_freq=h_freq, verbose=False)

                # Load events from the stim-track-generated events (ergo1)
                if EPOCHING_TRIGGER_SOURCE in ['ergo1', 'both']:
                    events_fpath = os.path.join(events_dir, f'sub-{sub_label}', f'sub-{sub_label}_task-{task_label}_run-{run_label}_events.tsv')
                    events_df = pd.read_csv(events_fpath, delimiter='\t') 
                    annot = mne.Annotations(onset=events_df.onset, duration=0.170, description=events_df.type)
                    data_filtered.set_annotations(annot)
                    events_from_annot_ergo1, event_dict2_ergo1 = mne.events_from_annotations(data_filtered)
                
                # Load events from the status channel
                if EPOCHING_TRIGGER_SOURCE in ['status', 'both']:
                    # loading events from the status events
                    events_path = bids_path.copy().update(suffix='events', extension='.tsv')
                    events_df = pd.read_csv(events_path.fpath, delimiter='\t')

                    # (optional) dropping events that are start less than 300ms from the previous
                    # diffs_onset = np.diff(events_df['onset'])
                    # too_close = np.where(diffs_onset < 0.3)[0] + 1
                    # events_df = events_df.drop(index=too_close).reset_index(drop=True)

                    annot = mne.Annotations(onset=events_df.onset, duration=0.170, description=events_df.trial_type)
                    data_filtered.set_annotations(annot)
                    events_from_annot_status, event_dict2_status = mne.events_from_annotations(data_filtered)
                
                # Epoch based on EPOCHING_TRIGGER_SOURCE
                epoch_kwargs = dict(on_missing='warn',
                                    picks=['Cz'],
                                    tmin=-1*bl_start, 
                                    tmax=0.4,
                                    baseline=[-1*bl_start, 0],
                                    reject=dict(eeg=75e-6),
                                    verbose=False)
                
                if EPOCHING_TRIGGER_SOURCE in ['ergo1', 'both']:
                    ergo1_epochs = mne.Epochs(data_filtered, 
                                        events=events_from_annot_ergo1, 
                                        event_id=event_dict2_ergo1, 
                                        **epoch_kwargs).drop_bad(verbose=False)
                    epoch_list_ergo1.append(ergo1_epochs)
                
                if EPOCHING_TRIGGER_SOURCE in ['status', 'both']:
                    status_epochs = mne.Epochs(data_filtered, 
                                        events=events_from_annot_status, 
                                        event_id=event_dict2_status, 
                                        **epoch_kwargs).drop_bad(verbose=False)
                    epoch_list_status.append(status_epochs)

            except Exception as e:
                print(f"No run {run_label} for task-{task_label}: {e}")
        
        # Concatenate and save epochs based on EPOCHING_TRIGGER_SOURCE
        if EPOCHING_TRIGGER_SOURCE in ['ergo1', 'both']:
            try:
                # combine epochs across runs
                all_epochs_ergo1 = mne.concatenate_epochs(epoch_list_ergo1)
            except IndexError as e:
                print(f'cannot run for sub-{sub_label} task-{task_label} run-{run_label}: {e}')

            try:
                out_base = f'sub-{sub_label}_task-{task_label}_baseline-{bl_start}_filt-{l_freq}-{h_freq}_run-all_event-stimtrack_epo.fif'
                all_epochs_ergo1.save(os.path.join(deriv_dir, out_base), overwrite=True)
            except Exception as e:
                print(f"Error saving epochs for sub-{sub_label} task-{task_label}: {e}")
                continue
        
        if EPOCHING_TRIGGER_SOURCE in ['status', 'both']:
            try:
                # combine epochs across runs
                all_epochs_status = mne.concatenate_epochs(epoch_list_status)
            except IndexError as e:
                print(f'cannot run for sub-{sub_label} task-{task_label} run-{run_label}: {e}')
            
            try:
                out_base = f'sub-{sub_label}_task-{task_label}_baseline-{bl_start}_filt-{l_freq}-{h_freq}_run-all_event-Status_epo.fif'
                all_epochs_status.save(os.path.join(deriv_dir_status, out_base), overwrite=True)
            except Exception as e:
                print(f"Error saving epochs for sub-{sub_label} task-{task_label}: {e}")
                continue


### Open-filter - not used in manuscript

In [ ]:

for sub_num in range(2, 35):
    sub_label = f'{sub_num:02d}'

    # initialize an empty dictionary for data
    task_evoked_dict = {}
    event_evoked_dict = {}

    for task_label in task_list:
        print(f'Loading {task_label} data')

        epoch_list = []

        for run_label in range(1,6):
        # load in EEG data
            bids_path = BIDSPath(root=bids_root, datatype='eeg', 
                                 subject=str(sub_label), 
                                 task=task_label, 
                                 run=run_label)
            
            try:
                data = read_raw_bids(bids_path)
                data.load_data()

            except:
                print(f"No run {run_label} for task-{task_label}")

            try:
                # re-reference data to linked mastoid reference
                data_ref = data.set_eeg_reference(ref_channels=['M1', 'M2'])
                
                # DO NOT filter data
                data_filtered = data_ref.copy()#.filter(l_freq=l_freq, h_freq=h_freq)


                # loading events from the stim-track-generated events
                events_fpath = sorted(glob(os.path.join(bids_root, 'derivatives', 'events-stimtrack') + 
                        f'/*{sub_label}*{task_label}*{run_label}_stimtrack_events.tsv'))[0]
                events_df = pd.read_csv(events_fpath, delimiter='\t') 
                annot = mne.Annotations(onset=events_df.onset, duration=0.170, description=events_df.type)
                data_filtered.set_annotations(annot)
                events_from_annot, event_dict2 = mne.events_from_annotations(data_filtered)
                
                # epoch data based on stimulus events
                epochs = mne.Epochs(data_filtered, 
                                    events=events_from_annot, #events, 
                                    event_id=event_dict2, #event_dict,
                                    on_missing='warn',
                                    picks=['Cz'],
                                    tmin=-1*bl_start, tmax=0.4, 
                                    baseline=[-1*bl_start, 0],
                                    reject=dict(eeg=75e-6)).drop_bad()
                
                epoch_list.append(epochs)
            except:
                print(f'Error occurred while processing sub-{sub_label} task-{task_label} run-{run_label}')

        try:
            # combine epochs across runs
            all_epochs = mne.concatenate_epochs(epoch_list)
            print(all_epochs)
        except:
            print(f'cannot combine epochs for sub-{sub_label} task-{task_label} run-{run_label}')

        try:
            out_base = f'sub-{sub_label}_task-{task_label}_baseline-{bl_start}_filt-open_run-all_event-stimtrack_epo.fif'
            all_epochs.save(os.path.join(deriv_dir, out_base), overwrite=True)
        except:
            continue


# Analyze whole-group data

Load stimtrack epochs

In [ ]:
act_pos_avgs = []
act_neg_avgs = []
act_eps = [sorted(glob(deriv_dir+f'/sub-{sub_num:02d}_task-active_baseline-{bl_start}_filt-{l_freq}-{h_freq}_run-all_event-stimtrack_epo.fif') for sub_num in range(2,35))]
for act_ep in act_eps[0]:
    # if any(f'sub-{sub_num:02d}' in act_ep for sub_num in [8, 10]):
    #     continue
    try:
        sub_active_epochs = mne.read_epochs(act_ep[0], verbose='WARNING')
        act_pos_avgs.append(sub_active_epochs['1'].average())
        act_neg_avgs.append(sub_active_epochs['2'].average())
    except:
        pass

pas_pos_avgs = []
pas_neg_avgs = []
pas_eps = [sorted(glob(deriv_dir+f'/sub-{sub_num:02d}_task-passive_baseline-{bl_start}_filt-{l_freq}-{h_freq}_run-all_event-stimtrack_epo.fif') for sub_num in range(2,35))]
for pas_ep in pas_eps[0]:
    # if any(f'sub-{sub_num:02d}' in pas_ep for sub_num in [8, 10]):
    #     continue
    try:
        sub_passive_epochs = mne.read_epochs(pas_ep[0], verbose='WARNING')
        pas_pos_avgs.append(sub_passive_epochs['1'].average())
        pas_neg_avgs.append(sub_passive_epochs['2'].average())
    except:
        pass
motor_avgs_st = []
motor_eps = sorted(glob(deriv_dir_motor + f'/sub-*_task-motor_baseline-{bl_start}_filt-{l_freq}-{h_freq}_run-all_event-Status_epo.fif'))
for motor_ep in motor_eps:
    try:
        sub_motor_epoch = mne.read_epochs(motor_ep, verbose=False)
        motor_avgs_st.append(sub_motor_epoch.average())
    except Exception as e:
        print(f"Error occurred: {e}")
        pass

Load status epochs - used for motor subtraction analysis. only using subjects 2-7.

In [ ]:
act_pos_avgs_st = []
act_neg_avgs_st = []
act_eps = sorted(glob(deriv_dir_status + f'/sub-*_task-active_baseline-{bl_start}_filt-{l_freq}-{h_freq}_run-all_event-Status_epo.fif'))
for act_ep in act_eps:
    if f'sub-08' in act_ep:
        break
    try:
        sub_active_epochs = mne.read_epochs(act_ep, verbose='WARNING')
        act_neg_avgs_st.append(sub_active_epochs['active/neg'].average())
        act_pos_avgs_st.append(sub_active_epochs['active/pos'].average())
    except:
        pass

pas_pos_avgs_st = []
pas_neg_avgs_st = []
pas_eps = sorted(glob(deriv_dir_status + f'/sub-*_task-passive_baseline-{bl_start}_filt-{l_freq}-{h_freq}_run-all_event-Status_epo.fif'))
for pas_ep in pas_eps:
    if f'sub-08' in pas_ep:
        break
    try:
        sub_passive_epochs = mne.read_epochs(pas_ep, verbose='WARNING')
        pas_neg_avgs_st.append(sub_passive_epochs['passive/neg'].average())
        pas_pos_avgs_st.append(sub_passive_epochs['passive/pos'].average())
    except:
        pass

motor_avgs_st = []
motor_eps = sorted(glob(deriv_dir_motor + f'/sub-*_task-motor_baseline-{bl_start}_filt-{l_freq}-{h_freq}_run-all_event-Status_epo.fif'))
for motor_ep in motor_eps:
    if f'sub-08' in motor_ep:
        break
    try:
        sub_motor_epoch = mne.read_epochs(motor_ep, verbose=False)
        motor_avgs_st.append(sub_motor_epoch.average())
    except Exception as e:
        print(f"Error occurred: {e}")
        pass

Average polarities within each subject

In [ ]:
if EPOCHING_TRIGGER_SOURCE in ['ergo1', 'both']:
    # stimtrack avgs
    passive_avgs = [mne.combine_evoked([pas_pos_avgs[x], pas_neg_avgs[x]], weights='nave')
                    for x in range(len(pas_pos_avgs))]
    active_avgs = [mne.combine_evoked([act_pos_avgs[x], act_neg_avgs[x]], weights='nave')
                    for x in range(len(act_pos_avgs))]

if EPOCHING_TRIGGER_SOURCE in ['status', 'both']:
    # status avgs
    passive_avgs_st = [mne.combine_evoked([pas_pos_avgs_st[x], pas_neg_avgs_st[x]], weights='nave')
                    for x in range(len(pas_pos_avgs_st))]
    active_avgs_st = [mne.combine_evoked([act_pos_avgs_st[x], act_neg_avgs_st[x]], weights='nave')
                    for x in range(len(act_pos_avgs_st))]

Grand averages

In [ ]:
if EPOCHING_TRIGGER_SOURCE in ['ergo1', 'both']:
    # stimtrack grand averages
    pas_pos_grandavg = mne.grand_average(pas_pos_avgs)
    pas_neg_grandavg = mne.grand_average(pas_neg_avgs)
    act_pos_grandavg = mne.grand_average(act_pos_avgs)
    act_neg_grandavg = mne.grand_average(act_neg_avgs)

    passive_grandavg = mne.grand_average(pas_pos_avgs+pas_neg_avgs)
    active_grandavg = mne.grand_average(act_pos_avgs+act_neg_avgs)

if EPOCHING_TRIGGER_SOURCE in ['status', 'both']:
    # status grand averages
    pas_pos_grandavg_st = mne.grand_average(pas_pos_avgs_st)
    pas_neg_grandavg_st = mne.grand_average(pas_neg_avgs_st)
    act_pos_grandavg_st = mne.grand_average(act_pos_avgs_st)
    act_neg_grandavg_st = mne.grand_average(act_neg_avgs_st)

    passive_grandavg_st = mne.grand_average(pas_pos_avgs_st+pas_neg_avgs_st)
    active_grandavg_st = mne.grand_average(act_pos_avgs_st+act_neg_avgs_st)

    motor_grandavg = mne.grand_average(motor_avgs_st)

Plotting stimtrack-generated waveform

In [ ]:
evokeds = dict(passive=pas_pos_avgs+pas_neg_avgs, active=act_pos_avgs+act_neg_avgs)
mne.viz.plot_compare_evokeds(evokeds, 
                             truncate_xaxis=False, 
                             picks='Cz', 
                             ci=0.95)

### Motor subtraction

In [ ]:
# subtract the motor ERP from the active and passive ERPs to isolate the auditory processing component
subtracted = mne.combine_evoked([active_grandavg_st, motor_grandavg], weights=[1, -1])
added = mne.combine_evoked([passive_grandavg_st, motor_grandavg], weights=[1, 1])

evokeds_dict_status = {
    'Active': active_grandavg_st,
    'Passive': passive_grandavg_st,
    'Active - Motor': subtracted,
    'Motor': motor_grandavg
}

# plot all (active, passive, motor, subtraction)
mne.viz.plot_compare_evokeds(
    evokeds_dict_status,
    picks='Cz',
    legend=True
)

In [ ]:
# create the subtracted waveform (by subject) -> so we can generate confidence intervals for the subtraction waveform
print(len(active_avgs_st), len(motor_avgs_st), len(passive_avgs_st))
subtracted = []
for i in range(len(active_avgs_st)):
    subtracted.append(mne.combine_evoked([active_avgs_st[i], motor_avgs_st[i]], weights=[1, -1]))

In [ ]:
# plot all status conditions with confidence intervals
evokeds = dict(active=active_avgs_st, passive=passive_avgs_st, motor=motor_avgs_st, subtracted=subtracted)
mne.viz.plot_compare_evokeds(evokeds, 
                             truncate_xaxis=False, 
                             picks='Cz', 
                             ci=0.95)

In [ ]:
# calculate peak latency and amplitude for all status conditions (passive, active, subtraction)
f_low=4
f_high=30
t_low=0.00
t_high=0.3

# Extract power for passive condition
pas_peak_lat_list = []
pas_peak_amp_list = []
for evk in passive_avgs_st:
    peak = evk.get_peak(tmin=t_low, tmax=t_high, return_amplitude=True)
    pas_peak_lat_list.append(peak[1])
    pas_peak_amp_list.append(peak[2])

# Extract power for active condition
act_peak_lat_list = []
act_peak_amp_list = []
for evk in active_avgs_st:
    peak = evk.get_peak(tmin=t_low, tmax=t_high, return_amplitude=True)
    act_peak_lat_list.append(peak[1])
    act_peak_amp_list.append(peak[2])


# subtracted
mot_peak_lat_list = []
mot_peak_amp_list = []
for evk in subtracted:
    peak = evk.get_peak(tmin=t_low, tmax=t_high, return_amplitude=True)
    mot_peak_lat_list.append(peak[1])
    mot_peak_amp_list.append(peak[2])

# Create a DataFrame for power values
peak_lat_dict = {'passive': pas_peak_lat_list,  
                 'active': act_peak_lat_list,
                 'motor': mot_peak_lat_list}

peak_amp_dict = {'passive': pas_peak_amp_list,  
                 'active': act_peak_amp_list,
                 'motor': mot_peak_amp_list}
peak_lat_df = pd.DataFrame(peak_lat_dict)
peak_amp_df = pd.DataFrame(peak_amp_dict)

In [ ]:
# latency stats
peak_pass_mean = peak_lat_df['passive'].mean()
peak_sub_mean = peak_lat_df['motor'].mean()
print(f'Mean passive peak latency: {peak_pass_mean:.03f} s')
print(f'Mean active - subtracted peak latency: {peak_sub_mean:.03f} s')
print(f'Mean active peak latency: {peak_lat_df["active"].mean():.03f} s')

# Perform paired t-test
t_stat, p_value = ttest_rel(peak_lat_df['passive'], peak_lat_df['motor'])
print(f"peak t-statistic: {t_stat:.03f}, peak p-value: {p_value:.05f}")

t_stat, p_value = ttest_rel(peak_lat_df['active'], peak_lat_df['motor'])
print(f"active t-statistic: {t_stat:.03f}, active p-value: {p_value:.05f}")

In [ ]:
# amplitude stats
peak_df = pd.DataFrame(peak_amp_dict)
print(peak_df.head())

# Compute the mean peak for each condition
peak_pass_mean = peak_df['passive'].mean()
peak_sub_mean = peak_df['motor'].mean()
print(f'Mean passive peak amplitude: {peak_pass_mean} ')
print(f'Mean active - subtracted peak amplitude: {peak_sub_mean}')

# Perform paired t-test
t_stat, p_value = ttest_rel(peak_df['passive'], peak_df['motor'])
print(f"peak t-statistic: {t_stat:.03f}, peak p-value: {p_value:.05f}")

### Trigger type ERP comparisions (plots, delays, SNR)

Debugging: plot each invdivual's evoked comparing between trigger types (not included in manuscript)

Subjects 2-7 have reliable ERPs across trigger types, but later subjects do not

In [ ]:
# plot each individual's evoked from ergo1 vs status channel - active
plt.close('all')
all_ids = list(range(2, 35))
skip_ids = [8, 10]
valid_ids = [s for s in all_ids if s not in skip_ids]

n = len(valid_ids)
ncols = 6
nrows = math.ceil(n / ncols)

fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(20, 10), sharey=True)
axes = axes.flatten()

for i, sub_id, stim_avg, status_avg, motor_avg in zip(range(len(valid_ids)), valid_ids, active_avgs, active_avgs_st, motor_avgs_st):
    ax = axes[i]
    ch_idx = stim_avg.ch_names.index('Cz')

    ax.plot(stim_avg.times,
            stim_avg.get_data()[ch_idx] * 1e6, 
            alpha=0.4, 
            linewidth=0.8,
            label='ergo1')
    ax.plot(status_avg.times,
             status_avg.get_data()[ch_idx] * 1e6, 
             alpha=0.4, 
             linewidth=0.8,
             label='status')
    ax.plot(motor_avg.times,
           motor_avg.get_data()[ch_idx] * 1e6,
           alpha=0.4,
           linewidth=0.8,
           label='motor')
    ax.set_title(f'Sub {sub_id}', fontsize=8)
    
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

# Shared axis labels
fig.text(0.5, 0.08, 'Time (s)', ha='center', fontsize=10)
fig.text(0.01, 0.5, 'Amplitude (µV)', va='center', rotation='vertical', fontsize=10)

# Single legend
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower right', fontsize=9)

fig.suptitle('Active — Cz: ergo1 vs status', fontsize=12, y=0.98)
plt.tight_layout()
plt.show()
     

In [ ]:
# plot each individual's evoked from ergo1 vs status channel - passive
plt.close('all')
all_ids = list(range(2, 35))
skip_ids = [8, 10]
valid_ids = [s for s in all_ids if s not in skip_ids]

n = len(valid_ids)
ncols = 6
nrows = math.ceil(n / ncols)

fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(20, 10), sharey=True)
axes = axes.flatten()

for i, sub_id, stim_avg, status_avg, motor_avg in zip(range(len(valid_ids)), valid_ids, passive_avgs, passive_avgs_st, motor_avgs_st):
    ax = axes[i]
    ch_idx = stim_avg.ch_names.index('Cz')

    ax.plot(stim_avg.times,
            stim_avg.get_data()[ch_idx] * 1e6, 
            alpha=0.4, 
            linewidth=0.8,
            label='ergo1')
    ax.plot(status_avg.times,
             status_avg.get_data()[ch_idx] * 1e6, 
             alpha=0.4, 
             linewidth=0.8,
             label='status')
    ax.plot(motor_avg.times,
           motor_avg.get_data()[ch_idx] * 1e6,
           alpha=0.4,
           linewidth=0.8,
           label='motor')
    ax.set_title(f'Sub {sub_id}', fontsize=8)
    
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

# Shared axis labels
fig.text(0.5, 0.08, 'Time (s)', ha='center', fontsize=10)
fig.text(0.01, 0.5, 'Amplitude (µV)', va='center', rotation='vertical', fontsize=10)

# Single legend
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower right', fontsize=9)

fig.suptitle('Passive — Cz: ergo1 vs status', fontsize=12, y=0.98)
plt.tight_layout()
plt.show()
     

Calculate delays between trigger types

In [ ]:
# first 6 subjects have a reliable delay between the ergo1 events and the status events
delayed_ids = np.arange(2, 8)
delays = []
for sub_ids, stim_avg, status_avg in zip(delayed_ids, active_avgs, active_avgs_st):
    ch_idx = stim_avg.ch_names.index('Cz')

    auto_corr = signal.correlate(stim_avg.get_data()[ch_idx], stim_avg.get_data()[ch_idx], mode='full')
    xcorr = signal.correlate(stim_avg.get_data()[ch_idx], status_avg.get_data()[ch_idx], mode='full')

    a1 = np.argmax(auto_corr)
    a2 = np.argmax(xcorr)
    delay = (a2 - a1) / stim_avg.info['sfreq']
    delays.append(delay)
    print(f'Subject {sub_ids}: Delay = {delay} seconds')

print(f'Mean delay across subjects: {np.mean(delays)} seconds')
print(f'Standard deviation of delay across subjects: {np.std(delays)} seconds')

In [ ]:
# subjects with high variable delay between the ergo1 events and the status events
all_ids = list(range(9, 35))
skip_ids = [8, 10]
valid_ids = [s for s in all_ids if s not in skip_ids]
delays = []
for sub_ids, stim_avg, status_avg in zip(valid_ids, active_avgs, active_avgs_st):
    ch_idx = stim_avg.ch_names.index('Cz')

    auto_corr = signal.correlate(stim_avg.get_data()[ch_idx], stim_avg.get_data()[ch_idx], mode='full')
    xcorr = signal.correlate(stim_avg.get_data()[ch_idx], status_avg.get_data()[ch_idx], mode='full')

    a1 = np.argmax(auto_corr)
    a2 = np.argmax(xcorr)
    delay = (a2 - a1) / stim_avg.info['sfreq']
    delays.append(delay)
    print(f'Subject {sub_ids}: Delay = {delay} seconds')

print(f'Mean delay across subjects: {np.mean(delays)} seconds')
print(f'Standard deviation of delays across subjects: {np.std(delays)} seconds')

#### SNR of each ERP for each trigger type

In [ ]:
# status
prestim_samples = int(abs(bl_start) * active_grandavg_st.info['sfreq'])
poststim_samples = int(0.300 * active_grandavg_st.info['sfreq'])

sub_ids = np.concatenate((np.arange(2, 8), [9], np.arange(11, 35)))

snr_list = []

for i, evk in enumerate(active_avgs_st):
    times = evk.times
    baseline_idx = np.where((times >= -1*bl_start) & (times < 0))[0]
    signal_idx = np.where((times >= 0) & (times <= 0.300))[0]
    noise_rms = np.sqrt(np.mean(evk.data[:, baseline_idx] ** 2, axis=1))
    signal_rms = np.sqrt(np.mean(evk.data[:, signal_idx] ** 2, axis=1))
    snr = 20 * np.log10(signal_rms / noise_rms)
    snr_list.append(snr[0])
    print(f'Subject {sub_ids[i]}  SNR: {snr.mean():.2f} dB  '
          f'pre={noise_rms.mean():.2e}  post={signal_rms.mean():.2e}')

In [ ]:
# ergo1
prestim_samples = int(abs(bl_start) * active_grandavg.info['sfreq'])
poststim_samples = int(0.300 * active_grandavg.info['sfreq'])

sub_ids = range(2, 35)

snr_list = []

for i, evk in enumerate(active_avgs):
    times = evk.times
    baseline_idx = np.where((times >= -1*bl_start) & (times < 0))[0]
    signal_idx = np.where((times >= 0) & (times <= 0.300))[0]
    noise_rms = np.sqrt(np.mean(evk.data[:, baseline_idx] ** 2, axis=1))
    signal_rms = np.sqrt(np.mean(evk.data[:, signal_idx] ** 2, axis=1))
    snr = 20 * np.log10(signal_rms / noise_rms)
    snr_list.append(snr[0])
    print(f'Subject {sub_ids[i]}  SNR: {snr.mean():.2f} dB  '
          f'pre={noise_rms.mean():.2e}  post={signal_rms.mean():.2e}')

In [ ]:
# motor
prestim_samples = int(abs(bl_start) * motor_grandavg.info['sfreq'])
poststim_samples = int(0.300 * motor_grandavg.info['sfreq'])

sub_ids = np.concatenate((np.arange(2, 8), [9], np.arange(11, 35)))

snr_list = []

for i, evk in enumerate(motor_avgs_st):
    times = evk.times
    baseline_idx = np.where((times >= -1*bl_start) & (times < 0))[0]
    signal_idx = np.where((times >= 0) & (times <= 0.300))[0]
    noise_rms = np.sqrt(np.mean(evk.data[:, baseline_idx] ** 2, axis=1))
    signal_rms = np.sqrt(np.mean(evk.data[:, signal_idx] ** 2, axis=1))
    snr = 20 * np.log10(signal_rms / noise_rms)
    snr_list.append(snr[0])
    print(f'Subject {sub_ids[i]}  SNR: {snr.mean():.2f} dB  '
          f'pre={noise_rms.mean():.2e}  post={signal_rms.mean():.2e}')

In [ ]:
# plot distribution of SNR values across subjects for the active condition
fig, ax = plt.subplots()
ax.hist(snr_list, bins=20, edgecolor='black')
ax.set_title('SNR Distribution for Active Condition')
ax.set_xlabel('SNR (dB)')
ax.set_ylabel('Number of Subjects')

### Peak stats - ergo1

In [ ]:
f_low=4
f_high=15
t_low=0.00
t_high=0.16

# Extract power for passive condition
pas_peak_lat_list = []
pas_peak_amp_list = []
for evk in passive_avgs:
    peak = evk.get_peak(tmin=t_low, tmax=t_high, return_amplitude=True)
    pas_peak_lat_list.append(peak[1])
    pas_peak_amp_list.append(peak[2])

# Extract power for active condition
act_peak_lat_list = []
act_peak_amp_list = []
for evk in active_avgs:
    peak = evk.get_peak(tmin=t_low, tmax=t_high, return_amplitude=True)
    act_peak_lat_list.append(peak[1])
    act_peak_amp_list.append(peak[2])

# Create a DataFrame for power values
peak_lat_dict = {'passive': pas_peak_lat_list,  
              'active': act_peak_lat_list}
peak_df = pd.DataFrame(peak_lat_dict)

In [ ]:
# Compute the mean peak for each condition
peak_pass_mean = peak_df['passive'].mean()
peak_act_mean = peak_df['active'].mean()
print(f'Mean passive peak latency: {peak_pass_mean:.03f} s')
print(f'Mean active peak latency: {peak_act_mean:.03f} s')

# Perform paired t-test
t_stat, p_value = ttest_rel(peak_df['passive'], peak_df['active'])
print(f"peak t-statistic: {t_stat:.03f}, peak p-value: {p_value:.05f}")

### Power stats - ergo1

In [ ]:
def compute_power(avg_evoked, f_low=90, f_high=110, t_low=0.1, t_high=0.2):
    # Compute the spectrogram (time-frequency representation)
    power = mne.time_frequency.tfr_multitaper(avg_evoked, 
                                            freqs=np.arange(f_low, f_high, 1), 
                                            n_cycles=2, 
                                            time_bandwidth=4.0,
                                            average=True, 
                                            return_itc=False)

    # Select the time and frequency bands
    time_mask = (power.times >= t_low) & (power.times <= t_high)
    freq_mask = (power.freqs >= f_low) & (power.freqs <= f_high)

    # Extract the power within the specified bands
    selected_power = power.data[:, freq_mask, :][:, :, time_mask]

    # Average power over the selected time and frequency bands
    average_power = selected_power.mean(axis=2).mean(axis=1)[0]
    return average_power

In [ ]:
f_low=4
f_high=30
t_low=-0.04
t_high=0.1

# Extract power for passive condition
pas_power_list = []
for evk in passive_avgs_st:
    power = compute_power(evk, f_low=f_low, f_high=f_high, t_low=t_low, t_high=t_high)
    pas_power_list.append(power)

# Extract power for active condition
act_power_list = []
for evk in active_avgs_st:
    power = compute_power(evk, f_low=f_low, f_high=f_high, t_low=t_low, t_high=t_high)
    act_power_list.append(power)

# Create a DataFrame for power values
power_dict = {'passive': pas_power_list,  
              'active': act_power_list}
power_df = pd.DataFrame(power_dict)

In [ ]:
# Compute the mean power for each condition
power_pass_mean = power_df['passive'].mean()
power_act_mean = power_df['active'].mean()
print(f'Mean passive power: {power_pass_mean}')
print(f'Mean active power: {power_act_mean}')

# Perform paired t-test
t_stat, p_value = ttest_rel(power_df['passive'], power_df['active'])
print(f"power t-statistic: {t_stat:.03f}, p-value: {p_value:.05f}")

#### Whole epoch

In [ ]:
f_low=4
f_high=30
t_low=-0.04
t_high=0.4

# Extract power for passive condition
pas_power_list = []
for evk in pas_pos_avgs+pas_neg_avgs:
    power = compute_power(evk, f_low=f_low, f_high=f_high, t_low=t_low, t_high=t_high)
    pas_power_list.append(power)

# Extract power for active condition
act_power_list = []
for evk in act_pos_avgs+act_neg_avgs:
    power = compute_power(evk, f_low=f_low, f_high=f_high, t_low=t_low, t_high=t_high)
    act_power_list.append(power)

# Create a DataFrame for power values
power_dict = {'passive': pas_power_list,  
              'active': act_power_list}
power_df = pd.DataFrame(power_dict)

In [ ]:
# Compute the mean power for each condition
power_pass_mean = power_df['passive'].mean()
power_act_mean = power_df['active'].mean()
print(f'Mean passive power: {power_pass_mean}')
print(f'Mean active power: {power_act_mean}')

# Perform paired t-test
t_stat, p_value = ttest_rel(power_df['passive'], power_df['active'])
print(f"power t-statistic: {t_stat:.03f}, p-value: {p_value:.03f}")

#### Compute grand average power

In [ ]:
# Extract power for each condition
f_low=4
f_high=15
t_low=0.1
t_high=0.2

passive_evoked_power = compute_power(passive_grandavg, 
                                     f_low=f_low, f_high=f_high, 
                                     t_low=t_low, t_high=t_high)

active_evoked_power = compute_power(active_grandavg, 
                                    f_low=f_low, f_high=f_high, 
                                    t_low=t_low, t_high=t_high)


#### Alpha band

In [ ]:
f_low=8
f_high=12
t_low=-0.04
t_high=0.4

# Extract power for passive condition
pas_power_list = []
for evk in passive_avgs:
    power = compute_power(evk, f_low=f_low, f_high=f_high, t_low=t_low, t_high=t_high)
    pas_power_list.append(power)

# Extract power for active condition
act_power_list = []
for evk in active_avgs:
    power = compute_power(evk, f_low=f_low, f_high=f_high, t_low=t_low, t_high=t_high)
    act_power_list.append(power)

# Create a DataFrame for power values
power_dict = {'passive': pas_power_list,  
              'active': act_power_list}
power_df = pd.DataFrame(power_dict)

In [ ]:
# Compute the mean power for each condition
power_pass_mean = power_df['passive'].mean()
power_act_mean = power_df['active'].mean()
print(f'Mean passive power: {power_pass_mean}')
print(f'Mean active power: {power_act_mean}')

# Perform paired t-test
t_stat, p_value = ttest_rel(power_df['passive'], power_df['active'])
print(f"power t-statistic: {t_stat:.03f}, p-value: {p_value:.03f}")

### frequency domain (spectra and spectrograms) - ergo1

In [ ]:
# Power spectra
passive_psd = passive_grandavg.compute_psd(tmin=0.0, tmax=0.300, fmin=1, fmax=30)
active_psd = active_grandavg.compute_psd(tmin=0.0, tmax=0.300, fmin=1, fmax=30)

# Spectrograms
frequencies = np.arange(4, 20)

passive_power = passive_grandavg.compute_tfr('multitaper', 
                                             freqs=frequencies, 
                                             n_cycles=1)
active_power = active_grandavg.compute_tfr('multitaper', 
                                           freqs=frequencies, 
                                           n_cycles=1)

### Plot Results

In [ ]:
# Manuscript figure 3: ERP waveforms, box plots for latency and power, spectrograms, and motor subtraction waveform
plt.close("all")
fig, axes = plt.subplot_mosaic(
    [["ERP", "ERP", "latency_box", "power_box"],
     ["p_spectrogram", "a_spectrogram", "diff_spectrogram", 'subtraction']],
     constrained_layout=True,
    #  width_ratios=[2, 2, 1, 1],
     dpi=150,
     figsize=(8, 6)
)

SMALL  = 11    # tick labels
MEDIUM = 12    # axis labels, colorbar label  
LARGE  = 13    # subplot titles

plt.rcParams.update({
    'font.size':        SMALL,
    'axes.titlesize':   LARGE,
    'axes.labelsize':   MEDIUM,
    'xtick.labelsize':  SMALL,
    'ytick.labelsize':  SMALL,
})

# set figure legend
fig.get_layout_engine().set(rect=[0, 0, 1, 0.92], hspace=0.0, wspace=0.0)
passive_patch = mpatches.Patch(color=passive_hex, label='Passive')
active_patch  = mpatches.Patch(color=active_hex,  label='Active')

fig.legend(
    handles=[passive_patch, active_patch],
    loc='upper center',
    ncols=2,
    frameon=False,
    fontsize=10,
)

# ERP waveform
mne.viz.plot_compare_evokeds(evokeds, 
                             truncate_xaxis=False, 
                             picks='Cz', 
                             ci=0.95,
                             axes=axes['ERP'],
                             colors={'passive': passive_hex, 'active': active_hex},
                             legend=False)
axes["ERP"].axvline(x=peak_pass_mean, color=passive_hex, linestyle='--', linewidth=1)
axes["ERP"].axvline(x=peak_act_mean, color=active_hex, linestyle='--', linewidth=1)
axes["ERP"].set_title("")
axes["ERP"].set_ylabel('Amplitude (µV)')
axes["ERP"].tick_params(direction="in")

# motor subtraction waveform
mne.viz.plot_compare_evokeds(
    evokeds_dict_status,
    picks='Cz',
    axes=axes['subtraction'],
    colors={'Passive': passive_hex, 'Active - Motor': 'k'},
    legend=False,
    linestyles=dict(Passive='-', **{'Active - Motor': '--'}),
    styles=dict(Active={"color": "#666666", 'alpha': 0.6}, Motor={"color": "#cccccc", 'alpha': 0.7}),
    truncate_xaxis=False,
    truncate_yaxis=False)
axes["subtraction"].set_title("")
axes["subtraction"].sharey(axes["ERP"])
axes["subtraction"].set_ylabel('')
axes["subtraction"].tick_params(direction="in")
axes["subtraction"].set_ylabel('Amplitude (µV)')

# note i editted the location of this legend post-hoc in inkscape to make it fit better, but this is the code for the legend that goes with the motor subtraction plot
handle_passive  = mlines.Line2D([], [], color=passive_hex,  linewidth=1, label="Passive")
handle_sub     = mlines.Line2D([], [], color="black", linewidth=1, linestyle=":",  label="Active - Motor")
handle_active = mlines.Line2D([], [], color="#666666",  linewidth=1, label="Active")
handle_motor = mlines.Line2D([], [], color="#cccccc", linewidth=1, label="Motor")

axes["subtraction"].legend(handles=[handle_passive, handle_sub, handle_active, handle_motor], loc="upper left", frameon=False, fontsize=7)

# box plot for power
sns.swarmplot(power_df,  
              linewidth=0.5, 
              log_scale=True,
              ax=axes["power_box"],
              size=2,
              palette=[passive_hex, active_hex],
              edgecolor='k')
sns.boxplot(power_df, 
            fliersize=0,
            ax=axes["power_box"],
            palette=[passive_hex, active_hex],
            linecolor='k')
axes["power_box"].set_ylabel(f'Power ({f_low}–{f_high} Hz; {t_low}–{t_high} s)')
axes["power_box"].set_yscale('log')
axes["power_box"].set_xlabel('')
axes["power_box"].set_xticklabels([])
axes["power_box"].tick_params(direction="in")
axes["power_box"].set_xticks([])
axes["power_box"].set_ylim(1e-10, 1e-7)

# box plot for P1 latency
t_low=0.00
t_high=0.16
sns.swarmplot(peak_df,  
              linewidth=0.5, 
              ax=axes["latency_box"],
              size=2,
              palette=[passive_hex, active_hex],
              edgecolor='k')
sns.boxplot(peak_df, 
            fliersize=0,
            ax=axes["latency_box"],
            palette=[passive_hex, active_hex],
            linecolor='k')

axes["latency_box"].set_ylabel(f'Latency ({t_low}–{t_high} s)')
axes["latency_box"].set_xlabel('')
axes["latency_box"].set_xticklabels([])
axes["latency_box"].tick_params(direction="in")
axes["latency_box"].set_xticks([])
axes["latency_box"].set_ylim(0.04, 0.2)


diff_power = active_power.copy()
diff_power.data = active_power.data - passive_power.data

passive_power.plot(axes=axes["p_spectrogram"], 
                   tmax=0.4,
                   colorbar=False, 
                   show=False,
                   vlim=[0, 9e-9],
                   cmap='magma')
active_power.plot(axes=axes["a_spectrogram"], 
                  tmax=0.4,
                  colorbar=False, 
                  show=False,
                  vlim=[0, 9e-9],
                  cmap='magma')

diff_power.plot(axes=axes["diff_spectrogram"],
                tmax=0.4,
                colorbar=False,
                cmap='RdBu_r',  
                show=False)

axes["p_spectrogram"].set_title('Passive')
axes["a_spectrogram"].set_title('Active')
axes["p_spectrogram"].sharex(axes["diff_spectrogram"])
axes["a_spectrogram"].sharex(axes["diff_spectrogram"])
axes["p_spectrogram"].set_ylabel('Frequency (Hz)')
axes["a_spectrogram"].set_ylabel('')
axes["p_spectrogram"].set_xlabel('Time (s)')
axes["a_spectrogram"].set_xlabel('Time (s)')
axes["p_spectrogram"].tick_params(direction="in")
axes["a_spectrogram"].tick_params(direction="in")
axes["diff_spectrogram"].set_title('Active - Passive')
axes["diff_spectrogram"].set_xlabel('Time (s)')
axes["diff_spectrogram"].set_ylabel('')
axes["diff_spectrogram"].tick_params(direction="in")

# add a single shared colorbar between the two spectrograms
im = axes["p_spectrogram"].images[0]  # grab the image from one of the axes
cbar = fig.colorbar(im, ax=[axes["p_spectrogram"], axes["a_spectrogram"]], 
                    location='bottom', 
                    shrink=0.6,
                    label='Power ($\\times 10^{-9}$)')

im_diff = axes["diff_spectrogram"].images[0]
cbar_diff = fig.colorbar(im_diff, 
                         ax=axes["diff_spectrogram"], 
                         location='bottom',  
                           label='Power difference ($\\times 10^{-9}$)')

# Disconnect all MNE's interactive callbacks
fig.canvas.mpl_disconnect_all() if hasattr(fig.canvas, 'mpl_disconnect_all') else None
for cid in list(fig.canvas.callbacks.callbacks.get('scroll_event', {}).keys()):
    fig.canvas.mpl_disconnect(cid)

fig.savefig('cortical.svg', dpi=600)
